# Blockchain Security & Smart Contract Auditor

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


## Project overview

**Objective:** Show how rule-based analysis can surface common smart-contract security smells for manual review.

**Workflow:** Parse representative contract content, apply focused security checks, classify findings, and summarize remediation guidance.

**Expected output:** Categorized findings with severity and recommended follow-up actions.

**Skills demonstrated:** Smart-contract security, static analysis, secure coding, rule design, Python

### Limitations

Pattern matching cannot prove a contract is safe. Production assessment requires compiler-aware static analysis, testing, and expert review.


In [ ]:
import hashlib
import json
import time
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import networkx as nx
from dataclasses import dataclass
from typing import List, Dict, Any
import re

@dataclass
class Transaction:
    sender: str
    recipient: str
    amount: float
    timestamp: float
    gas_used: int
    contract_address: str = None

class Block:
    def __init__(self, index, transactions, previous_hash, nonce=0):
        self.index = index
        self.timestamp = time.time()
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.nonce = nonce
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        block_string = json.dumps({
            "index": self.index,
            "timestamp": self.timestamp,
            "transactions": [tx.__dict__ for tx in self.transactions],
            "previous_hash": self.previous_hash,
            "nonce": self.nonce
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

class SmartContract:
    def __init__(self, address, code, creator):
        self.address = address
        self.code = code
        self.creator = creator
        self.balance = 0
        self.storage = {}
        self.call_count = 0

class BlockchainSecurityAnalyzer:
    def __init__(self):
        self.blockchain = []
        self.contracts = {}
        self.transaction_pool = []
        self.address_graph = nx.DiGraph()
        self.vulnerability_patterns = self.load_vulnerability_patterns()

    def load_vulnerability_patterns(self):
        return {
            "reentrancy": [
                r"call\.value\(\).*external_call",
                r"send\(.*\).*state_change"
            ],
            "integer_overflow": [
                r"uint\d+.*\+.*without.*SafeMath",
                r"balance\s*\+=.*without.*check"
            ],
            "access_control": [
                r"onlyOwner.*missing",
                r"modifier.*access.*undefined"
            ],
            "denial_of_service": [
                r"for\s*\(.*unbounded.*loop",
                r"while\s*\(.*gas_limit"
            ]
        }

    def create_genesis_block(self):
        genesis_tx = Transaction("genesis", "0x0", 0, time.time(), 0)
        genesis_block = Block(0, [genesis_tx], "0")
        self.blockchain.append(genesis_block)
        return genesis_block

    def add_block(self, transactions):
        previous_block = self.blockchain[-1]
        new_block = Block(len(self.blockchain), transactions, previous_block.hash)

        if self.validate_block(new_block):
            self.blockchain.append(new_block)
            self.update_address_graph(transactions)
            return new_block
        return None

    def validate_block(self, block):
        if len(self.blockchain) == 0:
            return True

        previous_block = self.blockchain[-1]
        return (block.previous_hash == previous_block.hash and
                block.index == previous_block.index + 1 and
                self.validate_transactions(block.transactions))

    def validate_transactions(self, transactions):
        for tx in transactions:
            if tx.amount < 0:
                return False
            if not self.validate_address(tx.sender) or not self.validate_address(tx.recipient):
                return False
        return True

    def validate_address(self, address):
        return len(address) == 42 and address.startswith("0x")

    def deploy_contract(self, code, creator):
        contract_address = self.generate_contract_address(creator)
        contract = SmartContract(contract_address, code, creator)
        self.contracts[contract_address] = contract

        deployment_tx = Transaction(creator, contract_address, 0, time.time(), 50000, contract_address)
        self.transaction_pool.append(deployment_tx)

        return contract_address

    def generate_contract_address(self, creator):
        nonce = len([tx for tx in self.get_all_transactions() if tx.sender == creator])
        address_data = f"{creator}{nonce}".encode()
        return "0x" + hashlib.sha256(address_data).hexdigest()[:40]

    def audit_smart_contract(self, contract_address):
        if contract_address not in self.contracts:
            return {"error": "Contract not found"}

        contract = self.contracts[contract_address]
        vulnerabilities = []

        for vuln_type, patterns in self.vulnerability_patterns.items():
            for pattern in patterns:
                if re.search(pattern, contract.code):
                    vulnerabilities.append({
                        "type": vuln_type,
                        "severity": self.assess_severity(vuln_type),
                        "pattern": pattern
                    })

        gas_analysis = self.analyze_gas_usage(contract)
        complexity_score = self.calculate_complexity(contract.code)

        return {
            "vulnerabilities": vulnerabilities,
            "gas_analysis": gas_analysis,
            "complexity_score": complexity_score,
            "recommendation": self.generate_recommendations(vulnerabilities)
        }

    def assess_severity(self, vuln_type):
        severity_map = {
            "reentrancy": "CRITICAL",
            "integer_overflow": "HIGH",
            "access_control": "MEDIUM",
            "denial_of_service": "HIGH"
        }
        return severity_map.get(vuln_type, "LOW")

    def analyze_gas_usage(self, contract):
        operations = {
            "SSTORE": 20000,
            "SLOAD": 200,
            "CALL": 700,
            "CREATE": 32000
        }

        estimated_gas = 21000
        for op, cost in operations.items():
            count = contract.code.count(op.lower())
            estimated_gas += count * cost

        return {
            "estimated_gas": estimated_gas,
            "optimization_potential": estimated_gas > 100000
        }

    def calculate_complexity(self, code):
        complexity_factors = {
            "loops": len(re.findall(r'\b(for|while)\b', code)) * 3,
            "conditions": len(re.findall(r'\b(if|else)\b', code)) * 2,
            "functions": len(re.findall(r'\bfunction\b', code)) * 1,
            "external_calls": len(re.findall(r'\.call\(', code)) * 4
        }
        return sum(complexity_factors.values())

    def generate_recommendations(self, vulnerabilities):
        recommendations = []

        for vuln in vulnerabilities:
            if vuln["type"] == "reentrancy":
                recommendations.append("Implement checks-effects-interactions pattern")
            elif vuln["type"] == "integer_overflow":
                recommendations.append("Use SafeMath library for arithmetic operations")
            elif vuln["type"] == "access_control":
                recommendations.append("Implement proper access control modifiers")
            elif vuln["type"] == "denial_of_service":
                recommendations.append("Limit loop iterations and gas consumption")

        return recommendations

    def detect_anomalous_transactions(self):
        all_transactions = self.get_all_transactions()

        amounts = [tx.amount for tx in all_transactions if tx.amount > 0]
        gas_usage = [tx.gas_used for tx in all_transactions]

        amount_threshold = np.percentile(amounts, 95) if amounts else 0
        gas_threshold = np.percentile(gas_usage, 95) if gas_usage else 0

        anomalies = []
        for tx in all_transactions:
            if tx.amount > amount_threshold or tx.gas_used > gas_threshold:
                anomalies.append({
                    "transaction": tx,
                    "reason": "High value/gas usage",
                    "risk_score": self.calculate_risk_score(tx)
                })

        return anomalies

    def calculate_risk_score(self, transaction):
        score = 0

        if transaction.amount > 1000:
            score += 30
        if transaction.gas_used > 100000:
            score += 25
        if transaction.contract_address:
            score += 20
        if self.is_suspicious_address(transaction.sender):
            score += 25

        return min(score, 100)

    def is_suspicious_address(self, address):
        tx_count = len([tx for tx in self.get_all_transactions() if tx.sender == address])
        return tx_count > 100 or address in ["0x" + "0" * 40]

    def analyze_transaction_flow(self):
        for tx in self.get_all_transactions():
            if tx.sender != "genesis":
                self.address_graph.add_edge(tx.sender, tx.recipient,
                                          weight=tx.amount, timestamp=tx.timestamp)

        centrality = nx.betweenness_centrality(self.address_graph)
        pagerank = nx.pagerank(self.address_graph)

        return {
            "centrality": centrality,
            "pagerank": pagerank,
            "suspicious_nodes": [addr for addr, score in centrality.items() if score > 0.1]
        }

    def get_all_transactions(self):
        transactions = []
        for block in self.blockchain:
            transactions.extend(block.transactions)
        return transactions

    def update_address_graph(self, transactions):
        for tx in transactions:
            if tx.sender != "genesis":
                self.address_graph.add_edge(tx.sender, tx.recipient, weight=tx.amount)

    def visualize_security_analysis(self):
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))

        block_times = []
        for i in range(1, len(self.blockchain)):
            time_diff = self.blockchain[i].timestamp - self.blockchain[i-1].timestamp
            block_times.append(time_diff)

        if block_times:
            axes[0, 0].plot(block_times)
            axes[0, 0].set_title('Block Creation Times')
            axes[0, 0].set_xlabel('Block Number')
            axes[0, 0].set_ylabel('Time Difference (s)')

        amounts = [tx.amount for tx in self.get_all_transactions() if tx.amount > 0]
        if amounts:
            axes[0, 1].hist(amounts, bins=20, alpha=0.7)
            axes[0, 1].set_title('Transaction Amount Distribution')
            axes[0, 1].set_xlabel('Amount')
            axes[0, 1].set_ylabel('Frequency')
            axes[0, 1].set_yscale('log')

        if len(self.address_graph.nodes()) > 0:
            pos = nx.spring_layout(self.address_graph, k=1, iterations=50)
            nx.draw(self.address_graph, pos, ax=axes[1, 0],
                   node_size=50, alpha=0.6, arrows=True)
            axes[1, 0].set_title('Transaction Flow Network')

        vulnerability_counts = defaultdict(int)
        for contract in self.contracts.values():
            audit_result = self.audit_smart_contract(contract.address)
            for vuln in audit_result["vulnerabilities"]:
                vulnerability_counts[vuln["type"]] += 1

        if vulnerability_counts:
            axes[1, 1].bar(vulnerability_counts.keys(), vulnerability_counts.values())
            axes[1, 1].set_title('Vulnerability Distribution')
            axes[1, 1].set_ylabel('Count')
            plt.setp(axes[1, 1].get_xticklabels(), rotation=45)

        plt.tight_layout()
        plt.show()

analyzer = BlockchainSecurityAnalyzer()
analyzer.create_genesis_block()

addresses = [f"0x{''.join([hex(np.random.randint(0,16))[2:] for _ in range(40)])}" for _ in range(20)]

for i in range(50):
    transactions = []
    for _ in range(np.random.randint(1, 5)):
        sender = np.random.choice(addresses)
        recipient = np.random.choice(addresses)
        amount = np.random.exponential(100)
        gas_used = np.random.randint(21000, 200000)
        tx = Transaction(sender, recipient, amount, time.time(), gas_used)
        transactions.append(tx)
    analyzer.add_block(transactions)

vulnerable_contract = """
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    msg.sender.call.value(amount)("");
    balances[msg.sender] -= amount;
}
for (uint i = 0; i < users.length; i++) {
    processUser(users[i]);
}
"""

secure_contract = """
function withdraw(uint amount) public {
    require(balances[msg.sender] >= amount);
    balances[msg.sender] = balances[msg.sender].sub(amount);
    require(msg.sender.send(amount));
}
"""

vuln_contract_addr = analyzer.deploy_contract(vulnerable_contract, addresses[0])
secure_contract_addr = analyzer.deploy_contract(secure_contract, addresses[1])

print(f"Blockchain length: {len(analyzer.blockchain)} blocks")
print(f"Total transactions: {len(analyzer.get_all_transactions())}")

vuln_audit = analyzer.audit_smart_contract(vuln_contract_addr)
print(f"Vulnerable contract audit - Vulnerabilities found: {len(vuln_audit['vulnerabilities'])}")
print(f"Complexity score: {vuln_audit['complexity_score']}")

secure_audit = analyzer.audit_smart_contract(secure_contract_addr)
print(f"Secure contract audit - Vulnerabilities found: {len(secure_audit['vulnerabilities'])}")

anomalies = analyzer.detect_anomalous_transactions()
print(f"Anomalous transactions detected: {len(anomalies)}")

flow_analysis = analyzer.analyze_transaction_flow()
print(f"Suspicious addresses identified: {len(flow_analysis['suspicious_nodes'])}")

analyzer.visualize_security_analysis()